### Setup (Unsloth Version)

This notebook is the **Unsloth** equivalent of `Notebook_1.ipynb`. It fine-tunes the same model (`microsoft/Phi-3-mini-4k-instruct`) on the same dataset (`dvgodoy/yoda_sentences`) using QLoRA, but leverages Unsloth for **2x faster training** and **~60% less memory**.

In [2]:
!pip install unsloth datasets huggingface-hub trl

  Using cached unsloth-2026.4.1-py3-none-any.whl.metadata (55 kB)
  Using cached unsloth_zoo-2026.4.2-py3-none-any.whl.metadata (32 kB)
  Using cached tyro-1.0.12-py3-none-any.whl.metadata (12 kB)
  Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
  Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
  Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (20 kB)
  Using cached cut_cross_entropy-25.1.1-py3-none-any.whl.metadata (9.3 kB)
  Using cached msgspec-0.20.0-cp312-cp312-manylinux2014_x86_64.m

### Imports

In [3]:
import os
import torch
from contextlib import nullcontext
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Loading a Quantized Base Model with Unsloth

With Unsloth, we don't need to manually set up `BitsAndBytesConfig`. The `FastLanguageModel.from_pretrained()` method handles 4-bit quantization automatically when `load_in_4bit=True` is set. It also applies optimized kernels for faster training and inference.

In [4]:
repo_id = "microsoft/Phi-3-mini-4k-instruct"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=repo_id,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,  # auto-detect: Float16 for Tesla T4/V100, Bfloat16 for Ampere+
)

==((====))==  Unsloth 2026.4.1: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

In [5]:
print(model.get_memory_footprint() / 1e6)

2206.341312


## Setting Up Low-Rank Adapters (LoRA) with Unsloth

Unsloth replaces `prepare_model_for_kbit_training()` + `get_peft_model()` with a single call to `FastLanguageModel.get_peft_model()`. This applies Unsloth's optimized LoRA kernels, which are **2x faster** than standard PEFT while producing identical results.

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,                   # the rank of the adapter
    lora_alpha=16,         # multiplier, usually 2*r
    lora_dropout=0.05,
    bias="none",
    target_modules=["o_proj", "qkv_proj", "gate_up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing (60% less VRAM)
    random_state=3407,
)

model

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: You added custom modules, but Unsloth hasn't optimized for this.
Beware - your finetuning might be noticeably slower!
Unsloth: You added custom modules, but Unsloth hasn't optimized for this.
Beware - your finetuning might be noticeably slower!


Unsloth 2026.4.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
              (k_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
              (v_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (defaul

In [7]:
print(model.get_memory_footprint() / 1e6)

2224.167104


In [8]:
trainable_parms, tot_parms = model.get_nb_trainable_parameters()
print(f'Trainable parameters:             {trainable_parms/1e6:.2f}M')
print(f'Total parameters:                 {tot_parms/1e6:.2f}M')
print(f'Fraction of trainable parameters: {100*trainable_parms/tot_parms:.2f}%')

Trainable parameters:             4.46M
Total parameters:                 3825.54M
Fraction of trainable parameters: 0.12%


## Formatting Your Dataset

Same dataset and formatting as the original notebook: `dvgodoy/yoda_sentences` with 720 English-to-Yoda-speak sentences, converted to the conversational format.

In [39]:
dataset = load_dataset("dvgodoy/yoda_sentences", split="train")
dataset

Dataset({
    features: ['sentence', 'translation', 'translation_extra'],
    num_rows: 720
})

In [40]:
dataset[0]

{'sentence': 'The birch canoe slid on the smooth planks.',
 'translation': 'On the smooth planks, the birch canoe slid.',
 'translation_extra': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}

In [41]:
# Convert dataset from prompt/completion format to conversational format
def format_dataset(examples):
    if isinstance(examples["prompt"], list):
        output_texts = []
        for i in range(len(examples["prompt"])):
            converted_sample = [
                {"role": "user", "content": examples["prompt"][i]},
                {"role": "assistant", "content": examples["completion"][i]},
            ]
            output_texts.append(converted_sample)
        return {"messages": output_texts}
    else:
        converted_sample = [
            {"role": "user", "content": examples["prompt"]},
            {"role": "assistant", "content": examples["completion"]},
        ]
        return {"messages": converted_sample}

# Formatting function required by Unsloth's SFTTrainer
# Must return a list of strings
def formatting_func(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(msgs, tokenize=False)
        texts.append(text)
    return texts

In [42]:
dataset = dataset.rename_column("sentence", "prompt")
dataset = dataset.rename_column("translation_extra", "completion")
dataset = dataset.map(format_dataset)
dataset = dataset.remove_columns(['prompt', 'completion', 'translation'])
messages = dataset[0]['messages']
messages

[{'content': 'The birch canoe slid on the smooth planks.', 'role': 'user'},
 {'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.',
  'role': 'assistant'}]

### Tokenizer

Unsloth already loaded the tokenizer for us. We just need to set the padding token.

In [43]:
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id

tokenizer.chat_template

"{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>\n' + message['content'] + '<|end|>\n'}}{% elif message['role'] == 'user' %}{{'<|user|>\n' + message['content'] + '<|end|>\n'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>\n' + message['content'] + '<|end|>\n'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>\n' }}{% else %}{{ eos_token }}{% endif %}"

In [44]:
print(tokenizer.apply_chat_template(messages, tokenize=False))

<|user|>
The birch canoe slid on the smooth planks.<|end|>
<|assistant|>
On the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|>
<|endoftext|>


## Fine-Tuning with SFTTrainer

The training configuration is the same as the original notebook. Unsloth is fully compatible with Hugging Face's `SFTTrainer` -- the speed and memory improvements come from the optimized model and LoRA kernels loaded earlier.

In [45]:
sft_config = SFTConfig(
    ## GROUP 1: Memory usage
    # Gradient checkpointing is handled by Unsloth's optimized version
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    # Gradient Accumulation / Batch size
    gradient_accumulation_steps=1,
    per_device_train_batch_size=16,
    auto_find_batch_size=True,
    ## GROUP 2: Training duration
    num_train_epochs=3,
    ## GROUP 3: Evaluation
    logging_steps=25,
    ## GROUP 4: Optimizer
    optim="adamw_8bit",   # Unsloth works great with 8-bit Adam
    learning_rate=2e-4,
    warmup_steps=0.1,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    ## GROUP 5: Precision
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    ## GROUP 6: Misc
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    output_dir="phi3-mini-yoda-unsloth",
    seed=42,
    report_to="none",
)

### SFTTrainer

In [46]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=dataset,
    formatting_func=formatting_func,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/720 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [47]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 720 | Num Epochs = 3 | Total steps = 135
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 4,456,448 of 3,825,536,000 (0.12% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,2.865704
50,1.570934
75,1.383323
100,1.322366
125,1.295890


TrainOutput(global_step=135, training_loss=1.6569595336914062, metrics={'train_runtime': 126.0462, 'train_samples_per_second': 17.137, 'train_steps_per_second': 1.071, 'total_flos': 1462555774457856.0, 'train_loss': 1.6569595336914062, 'epoch': 3.0})

## Querying the Model

Unsloth provides `FastLanguageModel.for_inference(model)` to switch to optimized inference mode (2x faster generation).

In [48]:
def gen_prompt(tokenizer, sentence):
    converted_sample = [
        {"role": "user", "content": sentence},
    ]
    prompt = tokenizer.apply_chat_template(converted_sample,
                                           tokenize=False,
                                           add_generation_prompt=True)
    return prompt

In [49]:
sentence = 'The Force is strong in you!'
prompt = gen_prompt(tokenizer, sentence)
print(prompt)

<|user|>
The Force is strong in you!<|end|>
<|assistant|>



In [50]:
# Switch to Unsloth's optimized inference mode
FastLanguageModel.for_inference(model)

tokenized_input = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)
generation_output = model.generate(
    **tokenized_input,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=64,
    use_cache=True,
)
decoded_output = tokenizer.decode(generation_output[0], skip_special_tokens=False)
print(decoded_output)

Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


<|user|> The Force is strong in you!<|end|><|assistant|> Strong in you, the Force is!<|end|><|endoftext|>


### Saving the Adapter

In [51]:
# Save just the LoRA adapter (small ~50MB)
model.save_pretrained("local-phi3-mini-yoda-adapter-unsloth")
tokenizer.save_pretrained("local-phi3-mini-yoda-adapter-unsloth")

('local-phi3-mini-yoda-adapter-unsloth/tokenizer_config.json',
 'local-phi3-mini-yoda-adapter-unsloth/chat_template.jinja',
 'local-phi3-mini-yoda-adapter-unsloth/tokenizer.json')

In [52]:
os.listdir("local-phi3-mini-yoda-adapter-unsloth")

['adapter_model.safetensors',
 'tokenizer_config.json',
 'tokenizer.json',
 'README.md',
 'adapter_config.json',
 'chat_template.jinja']

### Push to Hugging Face Hub (Optional)

In [55]:
from huggingface_hub import login
login()

In [57]:
# Push adapter to the Hub
trainer.push_to_hub()

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 15.0kB / 17.8MB            

  ...unsloth/training_args.bin:  60%|#####9    | 3.41kB / 5.71kB            

CommitInfo(commit_url='https://huggingface.co/ImShivu/phi3-mini-yoda-unsloth/commit/411c50b2950e93cf5b494a44d3ee671f4b620885', commit_message='End of training', commit_description='', oid='411c50b2950e93cf5b494a44d3ee671f4b620885', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ImShivu/phi3-mini-yoda-unsloth', endpoint='https://huggingface.co', repo_type='model', repo_id='ImShivu/phi3-mini-yoda-unsloth'), pr_revision=None, pr_num=None)

## Key Differences: Unsloth vs Original

| Aspect | Original (PyTorch + PEFT) | Unsloth |
|---|---|---|
| **Model loading** | `AutoModelForCausalLM` + `BitsAndBytesConfig` | `FastLanguageModel.from_pretrained(load_in_4bit=True)` |
| **LoRA setup** | `prepare_model_for_kbit_training()` + `get_peft_model()` | `FastLanguageModel.get_peft_model()` (single call) |
| **Gradient checkpointing** | Standard PyTorch | `use_gradient_checkpointing="unsloth"` (60% less VRAM) |
| **Inference** | Manual `model.eval()` + `autocast` | `FastLanguageModel.for_inference(model)` (2x faster) |
| **Training speed** | Baseline | ~2x faster (optimized kernels) |
| **Memory usage** | Baseline | ~60% less VRAM |
| **Trainer** | Same `SFTTrainer` | Same `SFTTrainer` (drop-in compatible) |